Script to use dataset from HuggingFace to create scenario dataset for prompt generation

In [1]:
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

load_dotenv(ROOT / "data" / ".env")

from datasets import Dataset, load_dataset

# Consumer health Q&A (Med-PaLM–style questions) + counseling chat scenarios
counsel_chat = load_dataset("nbertagnolli/counsel-chat")["train"]
healthsearchqa = load_dataset("katielink/healthsearchqa", "all_data")["train"]

README.md: 0.00B [00:00, ?B/s]

c:\Users\bi020\Documents\tcu\tcu-ds\research\honors\honores\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bi020\.cache\huggingface\hub\datasets--nbertagnolli--counsel-chat. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Repo card metadata block was not found. Setting CardData to empty.


20220401_counsel_chat.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2775 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

c:\Users\bi020\Documents\tcu\tcu-ds\research\honors\honores\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bi020\.cache\huggingface\hub\datasets--katielink--healthsearchqa. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


all.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/4436 [00:00<?, ? examples/s]

In [2]:
cc_df = counsel_chat.to_pandas()[["questionText"]].copy()
cc_df["source"] = "counsel_chat"

hq_df = healthsearchqa.to_pandas()[["question"]].rename(columns={"question": "questionText"})
hq_df["source"] = "healthsearchqa"

base_prompt = pd.concat([cc_df, hq_df], ignore_index=True)
base_prompt = base_prompt.dropna(subset=["questionText"])
base_prompt["questionText"] = base_prompt["questionText"].astype(str).str.strip()
base_prompt = base_prompt[base_prompt["questionText"] != ""]
base_prompt = base_prompt.drop_duplicates(subset=["questionText"])

base_prompts = Dataset.from_pandas(base_prompt, preserve_index=False)
len(base_prompts), base_prompt["source"].value_counts().to_dict()

(4021, {'healthsearchqa': 3156, 'counsel_chat': 865})

Use [HF transformer pipeline](https://huggingface.co/docs/transformers/en/main_classes/pipelines) to abstract model inference

In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())

True
4


In [4]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

model_name = "Qwen/Qwen3-4B-Instruct-2507"

# Load tokenizer with left padding
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token 
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
    
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


[HF prompting for text generation](https://huggingface.co/docs/transformers/en/tasks/prompting)

In [3]:
instruction = """
    Rewrite this base prompt into 4 variations based on 4 age groups to be used as input to a large language model (LLM). The variations should reflect how a user from each age group might phrase their question to the LLM. The age groups are:
    Teen (13–19)
    Young Adult (20–39)
    Middle-aged (40–65)
    Senior (65+)

    Guidelines:
    - Phrase the question by specifying user's age, describe their problem and symptoms relevant to base prompt, how long the problems or symptoms have been occurring, and their main concerns
    - Each variation should be a coherent question
    - Keep the scenario relevant to the base prompt
    - Make the age cue explicit but integrated naturally using a wide variety of ages in the defined range
    - Write in first person similar to how a human user would ask a question
    - Ensure the variations in each situation are consistent with the age range

    Output the generated variations in this JSON format:
    {
        "base_prompt": "Why is my heart beating fast?",
        "teen": "I am 16 and I've been experiencing a rapid heartbeat for the past few days when I'm at school. What could be causing this?",
        "young-adult": "I'm a 28-year-old and recently I've noticed my heart racing, especially when I'm stressed. It's been happening for about a week now. Should I be concerned?",
        "middle-aged": "I'm male, 45 years old and sometimes feel my heart racing even when doing nothing. I'm concerned about my heart health and would like to know what might be causing this.",
        "senior": "I'm 70 years old, not diagnosed with any heart condition but lately my heart seems to beat faster and it gets hard to catch my breath. What is wrong?"
    }
    """

# Add instruction to questions
def analyze_text(questions):
    questions = [q for q in questions if q is not None and str(q).strip() != ""]
    if not questions:   
        return []     
    
    messages = [
        [
            {"role": "system", "content": instruction},
            {"role": "user", "content": "Base prompt: " + question + "\n"}
        ]
        for question in questions
    ]
    output = generator(messages)
    return output

Use batch inference for augmentation to save time + GPU capacity. Suggest experimenting with batch size to get desirable results.

In [ ]:
outputs = []
batch_size = 100
count = 1

for i in range(0, len(base_prompts), batch_size):
    print("processing batch " + str(count) + "\n")
    batch = base_prompts["questionText"][i:i+batch_size]
    output = analyze_text(batch)
    count += 1
    outputs.append(output)

processing batch 1

processing batch 2

processing batch 3

processing batch 4

processing batch 5

processing batch 6



In [7]:
# each batch is a list of list of dictionary
len(outputs) # 10
len(outputs[0]) # 10
outputs[0][0][0] # {'generated_text':[{'system'}, {'user'}, {'assistant'}]}
outputs[0][0][0]['generated_text'][2]

{'role': 'assistant',
 'content': '{\n    "base_prompt": "I have known her for years. She was dating my brother-in-law when we met. My kids think of her as their aunt. On Halloween 2014, I lost my mom to cancer. My mom and dad were still married when she passed away. My friend was there for me through that and my own cancer diagnosis. She has been a very big part of both me and my kids’ life, but now last month, my dad told me that he really likes my friend and wants to marry her. She’s like a sister to me. My kids hate the idea.",\n    "variations": {\n        "teen": "I\'m 17 and my mom passed away in 2014, and I’ve known this girl for years—she was like a sister to me and even helped me through my own health struggles. My brother-in-law used to date her, and now my dad says he wants to marry her. I don’t get it—she’s been like family to me, and my siblings are really upset. What should I do?",\n        "young-adult": "I’m 32 and I’ve known this woman for over a decade. She was a hug

In [8]:
import json
import os

response = []
for batch in outputs:
    for item in batch:
        generated = item[0]["generated_text"][-1]  # assistant message
        content = generated.get("content", "")
        try:
            parsed = json.loads(content)
        except Exception:
            continue
        response.append(parsed)

os.makedirs(ROOT / "data", exist_ok=True)
with open(ROOT / "data" / "prompt_variations.json", "w", encoding="utf-8") as json_file:
    json.dump(response, json_file, indent=2)

Transform list of jsons into dataframe for better transformers inference

In [2]:
# Output of the previous cell is data/prompt_variations.json under the project root (one JSON list).
# If the model nests age variants under "variations", flatten before reading fields.

import json
import pandas as pd


def _flatten_variations(q: dict) -> dict:
    if not isinstance(q, dict):
        return q
    merged = {k: v for k, v in q.items() if k != "variations"}
    if isinstance(q.get("variations"), dict):
        merged.update(q["variations"])
    return merged


with open(ROOT / "data" / "prompt_variations.json", "r", encoding="utf-8") as f:
    data = json.load(f)

question_list = []
age_group = []

for q in data:
    q = _flatten_variations(q)
    question_list.append(q["base_prompt"])
    question_list.append(q["teen"])
    question_list.append(q["young-adult"])
    question_list.append(q["middle-aged"])
    question_list.append(q["senior"])
    age_group.append("base_prompt")
    age_group.append("teen")
    age_group.append("young_adult")
    age_group.append("middle_aged")
    age_group.append("senior")

prompts = list(zip(question_list, age_group))
df = pd.DataFrame(prompts, columns=["prompt", "age_group"])

In [3]:
len(df)

20145

In [4]:
df.to_csv(ROOT / "data" / "gen_prompts.csv", index=False)